## *Phase 4 — Window Functions*

---
*Number all orders chronologically.*

In [0]:
%sql
SELECT
  Order_Number,
  Order_Date,
  row_number() OVER (
    --PARTITION BY Order_Number
    ORDER BY Order_Date) AS Order_Row_Number,
  rank() OVER (
    --PARTITION BY Order_Number
    ORDER BY Order_Date) AS Order_Rank,
  dense_rank() OVER (
    --PARTITION BY Order_Number 
    ORDER BY Order_Date) AS Order_dense_rank
FROM
  bara_slaes_project.gold.factable;

---
*Number products inside each category.*

In [0]:
%sql
SELECT
  Category_Name,
  Product_name,
  row_number() OVER (
      PARTITION BY Category_Name
      ORDER BY Category_Name
    ) AS Number_Of_Products__Per_Category
FROM
  bara_slaes_project.gold.dimproduct_info;

---
*Number customer purchases.*

In [0]:
%sql
SELECT
  Customer_ID,
  Order_Number,
  Order_Date,
  --Sales,
  --sum(Sales) over(PARTITION BY Customer_ID ORDER BY Customer_ID) TotalSalesPerCustomer,
  --avg(Sales) over(PARTITION BY Customer_ID ORDER BY Customer_ID) AvgSalesPerCustomer,
  row_number() OVER (
      PARTITION BY Customer_ID
      ORDER BY Order_Date
    ) AS Number_Of_Orders
FROM
  bara_slaes_project.gold.factable As Fac
    LEFT JOIN bara_slaes_project.gold.dimcustomer_info As cus
      ON Fac.Customer_srgk = cus.Customer_srgk;

---
*Rank products by sales.*

In [0]:
%sql
SELECT
  prod.Product_name,
  fac.Sales,
   rank() OVER (
      ORDER BY Fac.Sales DESC
    ) AS RankProductsBySales
FROM
  bara_slaes_project.gold.factable AS Fac
    LEFT JOIN bara_slaes_project.gold.dimproduct_info Prod
      ON Fac.Product_srgk = Prod.Product_srgk;

---
*Rank country by revenue.*

In [0]:
%sql
SELECT
  Cus.Country,
  fac.Sales,
   rank() OVER (
      ORDER BY Fac.Sales DESC
    ) AS RankCountryByRevenue
FROM
  bara_slaes_project.gold.factable AS Fac
    LEFT JOIN bara_slaes_project.gold.dimcustomer_info Cus
      ON Fac.Product_srgk = Cus.Customer_srgk;

---
*Rank Gender by Sales.*

In [0]:
%sql
SELECT
  Cus.Gender,
  fac.Sales,
   rank() OVER (
      ORDER BY Fac.Sales DESC
    ) AS RankGenderBySales
FROM
  bara_slaes_project.gold.factable AS Fac
    LEFT JOIN bara_slaes_project.gold.dimcustomer_info Cus
      ON Fac.Product_srgk = Cus.Customer_srgk;

---
*Divide customers into four spending groups.*

In [0]:
%sql
SELECT
  Cus.Customer_ID,
  fac.Sales,
   ntile(4) OVER (
      PARTITION BY Cus.Customer_ID
      ORDER BY Cus.Customer_ID DESC
    ) AS CustomerSpendingGroups
FROM
  bara_slaes_project.gold.factable AS Fac
    LEFT JOIN bara_slaes_project.gold.dimcustomer_info Cus
      ON Fac.Product_srgk = Cus.Customer_srgk;
    

---
*Divide products into five price categories.*

In [0]:
%sql
SELECT
    Prod.Product_ID, 
    Prod.Product_name,
    fac.Price,
    ntile(5) OVER (
        PARTITION BY Prod.Product_ID
        ORDER BY Prod.Product_ID DESC
        ) AS ProductsPriceCategories
FROM
  bara_slaes_project.gold.factable AS Fac
    LEFT JOIN bara_slaes_project.gold.dimproduct_info AS Prod
      ON Fac.Product_srgk = Prod.Product_srgk
ORDER BY ProductsPriceCategories  DESC   ;
    

---
*Compare today's sales with previous day's sales.*

In [0]:
%sql
SELECT
  Order_Date,
  Sales,
  lag(Sales) OVER (ORDER BY Order_Date) As PreviousDaySales,
  lag(Sales,2) OVER (ORDER BY Order_Date) As Previous2DaySales
FROM
  bara_slaes_project.gold.factable;

---
*Predict next shipment date.*

In [0]:
%sql
SELECT
  Ship_Date,
  lead(Ship_Date,3) OVER (ORDER BY Order_Date) As Next3DyasShipmentDate
FROM
  bara_slaes_project.gold.factable;

---
*Compare current order with next customer order.*

In [0]:
%sql
SELECT
  CUS.Customer_ID,
  cus.Customer_first_name,
  Order_Date,
  lead(Order_Date) OVER (PARTITION BY CUS.Customer_ID ORDER BY Order_Date) AS NextCustomerOrderDate
FROM
  bara_slaes_project.gold.factable as fac
    LEFT JOIN bara_slaes_project.gold.dimcustomer_info Cus
      ON fac.Customer_srgk = Cus.Customer_srgk;

---
*Show first order placed by every customer.*

In [0]:
%sql
SELECT
  CUS.Customer_ID,
  fac.order_number,
  cus.Customer_first_name,
  fac.order_date,
  first_value(fac.order_number) OVER (PARTITION BY CUS.Customer_ID ORDER BY fac.order_date) AS FirstOrderPlacedByCustomer
FROM
  bara_slaes_project.gold.factable as fac
    LEFT JOIN bara_slaes_project.gold.dimcustomer_info Cus
      ON fac.Customer_srgk = Cus.Customer_srgk;

---
*Show latest purchase by every customer.*

In [0]:
%sql
SELECT
  CUS.Customer_ID,
  fac.order_number,
  cus.Customer_first_name,
  fac.order_date,
  last_value(fac.order_number) OVER (
      PARTITION BY CUS.Customer_ID
      ORDER BY fac.order_date
      ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS LastOrderPlacedByCustomer
FROM
  bara_slaes_project.gold.factable as fac
    LEFT JOIN bara_slaes_project.gold.dimcustomer_info Cus
      ON fac.Customer_srgk = Cus.Customer_srgk;

---
*Running Total  Sales*

In [0]:
%sql
select
  order_date,
  Sales,
  SUM(Sales) OVER (
      ORDER BY order_date
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS RunningTtotalSales
from
  bara_slaes_project.gold.factable;

---
*Running customer purchases -sales*

In [0]:
%sql
select
  fac.order_date,
  cus.customer_id,
  cus.customer_first_name,
  fac.Sales,
  SUM(fac.Sales) OVER (
      PARTITION BY cus.customer_id
      ORDER BY fac.order_date
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS RunningTtotalSalesPeRcustomer
from
  bara_slaes_project.gold.factable as fac
  left join bara_slaes_project.gold.dimcustomer_info as cus
    on fac.customer_srgk = cus.customer_srgk;

---
*Moving Average*

In [0]:
%sql
select
  order_date,
  Sales,
  avg(Sales) OVER (
      ORDER BY order_date
      ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS MovingAverageSales,
    round((MovingAverageSales-Sales)/MovingAverageSales,3) as MovingAverageSalesPercentageChange
from
  bara_slaes_project.gold.factable;

---
*Seven-day moving average of sales.*

In [0]:
%sql
select
  order_date,
  Sales,
  avg(Sales) OVER (
      ORDER BY order_date
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS 7DaysMovingAverageSales,
    round((7DaysMovingAverageSales-Sales)/7DaysMovingAverageSales,3) as 7DaysMovingAverageSalesPercentageChange
from
  bara_slaes_project.gold.factable;

---
*Thirty-day moving average.*

In [0]:
%sql
select
  order_date,
  Sales,
  avg(Sales) OVER (
      ORDER BY order_date
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS 30DaysMovingAverageSales,
    round((30DaysMovingAverageSales-Sales)/30DaysMovingAverageSales,3) as 30DaysMovingAverageSalesPercentageChange
from
  bara_slaes_project.gold.factable;